In [ ]:
import numpy as np
import h5py
import os
import pandas as pd
import kipoiseq

In [20]:
DIR = "/eagle/AIHPC4Edu/ssalazar/projects/enformer_training/basenji_data_h5/"
seqs_file = os.path.join(DIR, "test_pop_seq.hdf5")
DIR_freqs = "/eagle/AIHPC4Edu/ssalazar/projects/enformer_training/EUR_allele_freqs_test"

In [58]:
with h5py.File(seqs_file, 'a') as h5f:
    del h5f['pop_sequence']
    print(h5f.keys())

<KeysViewHDF5 ['large_sequence', 'query_regions', 'sequence', 'target']>


In [12]:
base_to_index = {'A': 0, 'C': 1, 'G': 2, 'T': 3}


# test code

In [52]:
with h5py.File(seqs_file, 'r') as h5f:
    print(h5f.keys())

<KeysViewHDF5 ['large_sequence', 'query_regions', 'sequence', 'target']>


In [39]:
with h5py.File(seqs_file, 'r') as h5f:
    chr, start, end = h5f['query_regions'][777, :]

    if chr == 0:
            chr = 'X'
    else:
        chr = int(chr)

    ref_seq = h5f['large_sequence'][777, :]

    expanded_interval = kipoiseq.Interval(chrom=f'chr{chr}',
                                          start=int(start),
                                          end=int(end)).resize(393216)

coordinates = np.arange(expanded_interval.start + 1, expanded_interval.start + 1 + 393216) 

In [17]:
expanded_interval

Interval(chrom='chr19', start=9074795, end=9468011, name='', strand='.', ...)

In [49]:
coordinates[37:39]

array([9074833, 9074834])

In [ ]:
ref_seq[37:39] # TG

array([[0., 0., 0., 1.],
       [0., 0., 1., 0.]], dtype=float32)

In [21]:
dosages = pd.read_csv(os.path.join(DIR_freqs, f'{expanded_interval.chrom}_EUR_allele_freqs.txt'), sep=' ', header=None)
dosage_series = pd.Series(dosages[3].values, index=dosages[1])
aligned_dosages = dosage_series.reindex(coordinates, fill_value=0)

In [24]:
dosages[2] = dosages[2].map(base_to_index) # map dosage values to second dimension depending on base
allele_series = pd.Series(dosages[2].values, index=dosages[1])
aligned_alleles = allele_series.reindex(coordinates, fill_value=-1).values # vector of -1s and alternate allele position as index of second dimension

mask = aligned_alleles != -1
rows = np.where(mask)[0] # which nucleotide positions need updating [10, 22] (positions)
alt_cols = aligned_alleles[mask] # the column index of the alternate allele [2, 3] G -> 2, T -> 3
alt_vals = aligned_dosages[mask] # the alterante allele dosage [0.2, 0.6] 
ref_cols = ref_seq[rows].argmax(axis=1) # get the column index where the reference base was


In [25]:
result = ref_seq
result[rows, ref_cols] = 1.0 - alt_vals # reduce reference base probability
result[rows, alt_cols] = alt_vals

In [43]:
diff_indices = np.where(np.any(ref_seq != result, axis=1))[0]

In [ ]:
result[37:39] # T > C YES

array([[0.      , 0.199612, 0.      , 0.800388],
       [0.      , 0.      , 1.      , 0.      ]], dtype=float32)

# run code

In [50]:
def get_popseq(interval, ref_seq, DIR_freqs, length):
    '''
    start: kipoi expanded interval
    ref_seq: one-hot-encoded reference sequence
    DIR_freqs: directory path where allele dosages per chromosome are stored
    '''
    coordinates = np.arange(interval.start + 1, interval.start + 1 + length) # 393_216 long vector of genomic coordinates

    dosages = pd.read_csv(os.path.join(DIR_freqs, f'{interval.chrom}_EUR_allele_freqs.txt'), sep=' ', header=None)
    dosage_series = pd.Series(dosages[3].values, index=dosages[1])
    aligned_dosages = dosage_series.reindex(coordinates, fill_value=0) # vector of zeroes and alternative allele dosages of length 393_216
    
    dosages[2] = dosages[2].map(base_to_index) # map dosage values to second dimension depending on base
    allele_series = pd.Series(dosages[2].values, index=dosages[1])
    aligned_alleles = allele_series.reindex(coordinates, fill_value=-1).values # vector of -1s and alternate allele position as index of second dimension

    mask = aligned_alleles != -1
    rows = np.where(mask)[0] # which nucleotide positions need updating [10, 22] (positions)
    alt_cols = aligned_alleles[mask] # the column index of the alternate allele [2, 3] G -> 2, T -> 3
    alt_vals = aligned_dosages[mask] # the alterante allele dosage [0.2, 0.6] 
    ref_cols = ref_seq[rows].argmax(axis=1) # get the column index where the reference base was

    result = ref_seq
    result[rows, ref_cols] = 1.0 - alt_vals # reduce reference base probability
    result[rows, alt_cols] = alt_vals  # assign alt base probability
    # result = np.zeros((length, 4), dtype=aligned_dosages.dtype)
    # rows = np.arange(length)
    # cols = aligned_alleles
    # result[rows, cols] = aligned_dosages # zeroes and allele dosages values in correct position
    # mask = (result != 0).any(axis=1)
    # ref_seq[mask] = result[mask] # replace the non-zero allele dosages on reference one hot encoded sequence
    return result

In [59]:
with h5py.File(seqs_file, 'r+') as h5f:
    num_seqs = h5f['sequence'].shape[0]

    # Create dataset if it doesn't exist yet
    if 'pop_sequence' not in h5f:
        h5f.create_dataset(
            'pop_sequence',
            shape=(0, 393216, 4),
            maxshape=(None, 393216, 4),
            dtype=np.float32
        )

    for n_seq in range(num_seqs):
        if n_seq % 30 == 0: 
            print(f"{n_seq+1}/{(num_seqs)}")
        chr, start, end = h5f['query_regions'][n_seq, :]
        if chr == 0:
            chr = 'X'
        else:
            chr = int(chr)
        ref_seq = h5f['large_sequence'][n_seq, :]

        expanded_interval = kipoiseq.Interval(
            chrom=f'chr{chr}',
            start=int(start),
            end=int(end)
        ).resize(393216)

        pop_seq = get_popseq(expanded_interval, ref_seq=ref_seq, DIR_freqs=DIR_freqs, length=393216)

        # Resize the dataset to fit new entry
        h5f['pop_sequence'].resize((n_seq + 1, 393216, 4))

        # Write the pop_seq into the dataset
        h5f['pop_sequence'][n_seq, :, :] = pop_seq


1/1937
31/1937
61/1937
91/1937
121/1937
151/1937
181/1937
211/1937
241/1937
271/1937
301/1937
331/1937
361/1937
391/1937
421/1937
451/1937
481/1937
511/1937
541/1937
571/1937
601/1937
631/1937
661/1937
691/1937
721/1937
751/1937
781/1937
811/1937
841/1937
871/1937
901/1937
931/1937
961/1937
991/1937
1021/1937
1051/1937
1081/1937
1111/1937
1141/1937
1171/1937
1201/1937
1231/1937
1261/1937
1291/1937
1321/1937
1351/1937
1381/1937
1411/1937
1441/1937
1471/1937
1501/1937
1531/1937
1561/1937
1591/1937
1621/1937
1651/1937
1681/1937
1711/1937
1741/1937
1771/1937
1801/1937
1831/1937
1861/1937
1891/1937
1921/1937


In [10]:
with h5py.File(val_seqs, 'r') as h5f:
    region = h5f['query_region'][231,:]
    popseq = h5f['pop_sequence'][231,:]
    refseq = h5f['sequence'][231,:]

In [12]:
int(region[2])

230419611

232:chr2_230288539_230419611

chr2	230157468	230550684 expanded

In [13]:
import kipoiseq

In [14]:
chr = f'chr{int(region[0])}'
start = int(region[1])
end = int(region[2])
expanded_interval = kipoiseq.Interval(
            chrom=chr,
            start=int(start),
            end=int(end)
        ).resize(393_216)

In [15]:
expanded_interval 

Interval(chrom='chr2', start=230157467, end=230550683, name='', strand='.', ...)

In [16]:
coordinates = np.arange(expanded_interval.start + 1, expanded_interval.start + 1 + 393_216) # 393_216 long vector of genomic coordinates

In [17]:
coordinates[13400:13411]

array([230170868, 230170869, 230170870, 230170871, 230170872, 230170873,
       230170874, 230170875, 230170876, 230170877, 230170878])

In [18]:
popseq[13400:13411,:]

array([[1.      , 0.      , 0.      , 0.      ],
       [0.      , 1.      , 0.      , 0.      ],
       [0.      , 1.      , 0.      , 0.      ],
       [1.      , 0.      , 0.      , 0.      ],
       [0.      , 0.      , 0.      , 1.      ],
       [0.      , 0.108527, 0.      , 0.891473],
       [0.      , 0.      , 0.      , 1.      ],
       [0.      , 0.      , 1.      , 0.      ],
       [1.      , 0.      , 0.      , 0.      ],
       [0.      , 1.      , 0.      , 0.      ],
       [0.      , 1.      , 0.      , 0.      ]], dtype=float32)

In [ ]:
refseq[13400:13411,:]

array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.]], dtype=float32)

chr2 230170873 T>C 0.108527